# Algorithm 5 — Latency-Utility Model

**File:** `src/CopilotScope.Collector/Quality/Analyzers.cs` (lines 92–132)

Maps raw time-to-first-token (TTFT) measurements to a **user-perceived utility** on $[0, 1]$
using a log-linear curve grounded in HCI research on response-time perception.

---

## 1  Utility Function

$$
U(t) =
\begin{cases}
  1 & t \leq 300\,\text{ms} \\
  0 & t \geq 10{,}000\,\text{ms} \\
  1 - \dfrac{\ln(t / 300)}{\ln(10000 / 300)} & \text{otherwise}
\end{cases}
$$

Key landmarks:
- **300 ms** — perceptual immediacy boundary (below: feels instant)
- **2 000 ms** — attention degradation threshold
- **8 000 ms** — abandonment risk threshold
- **10 000 ms** — utility floor (fully breaks flow)

The log scale reflects that humans perceive latency changes *multiplicatively* (a 1 s
increase at 9 s is far less noticeable than at 0.5 s).

---

## 2  Session-level Metrics

Applied **per sample** $t_i$, not just to the p50:

$$
\bar{U} = \frac{1}{N}\sum_{i=1}^{N} U(t_i)
$$

Percentiles are computed on the raw TTFT distribution:

$$
p50 = \text{Percentile}(\{t_i\}, 0.5), \quad p95 = \text{Percentile}(\{t_i\}, 0.95)
$$

Risk fractions:

$$
f_{\text{degraded}} = \frac{|\{i : t_i > 2000\}|}{N}, \qquad
f_{\text{abandon}} = \frac{|\{i : t_i > 8000\}|}{N}
$$

The score returned to the pipeline is $\bar{U}$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def utility(ms: float) -> float:
    if ms <= 300:    return 1.0
    if ms >= 10_000: return 0.0
    return 1.0 - np.log(ms / 300.0) / np.log(10_000.0 / 300.0)

def percentile(values: list[float], p: float) -> float:
    if not values: return 0.0
    s = sorted(values)
    idx = int(np.ceil(p * len(s))) - 1
    return s[max(0, min(idx, len(s) - 1))]

def latency_report(ttft_samples: list[float]) -> dict:
    N = len(ttft_samples)
    if N == 0: return {}
    utils = [utility(t) for t in ttft_samples]
    mean_u = np.mean(utils)
    p50 = percentile(ttft_samples, 0.50)
    p95 = percentile(ttft_samples, 0.95)
    degraded = sum(1 for t in ttft_samples if t > 2000) / N
    abandon  = sum(1 for t in ttft_samples if t > 8000) / N
    return {'mean_utility': mean_u, 'p50': p50, 'p95': p95,
            'f_degraded': degraded, 'f_abandon': abandon}

# Key landmark values
print('=== Utility at landmark TTFT values ===')
for ms in [100, 300, 500, 1000, 2000, 4000, 8000, 10000]:
    print(f'  TTFT {ms:>6} ms → U = {utility(ms):.4f}')

# Session scenarios
rng = np.random.default_rng(42)
scenarios = {
    'Fast model  (lognormal μ=6.0, σ=0.3)':    list(rng.lognormal(6.0, 0.3, 50)),
    'Medium model (lognormal μ=7.0, σ=0.4)':   list(rng.lognormal(7.0, 0.4, 50)),
    'Slow model  (lognormal μ=8.0, σ=0.5)':    list(rng.lognormal(8.0, 0.5, 50)),
    'Bi-modal (fast+spike)':                    list(rng.lognormal(6.0, 0.3, 40)) +
                                                 list(rng.lognormal(9.0, 0.3, 10)),
}

print(f'\n{"Scenario":<40} {"meanU":>7} {"p50 ms":>8} {"p95 ms":>8} {"degraded":>9} {"abandon":>8}')
print('-' * 90)
for name, samples in scenarios.items():
    r = latency_report(samples)
    print(f'{name:<40} {r["mean_utility"]:>7.3f} {r["p50"]:>8.0f} {r["p95"]:>8.0f} '
          f'{r["f_degraded"]:>9.1%} {r["f_abandon"]:>8.1%}')

In [ ]:
t = np.logspace(np.log10(100), np.log10(12000), 400)
u = np.array([utility(v) for v in t])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(t, u, lw=2.5, color='steelblue', label='$U(t)$')
ax.axvline(300,    ls='--', color='green',  alpha=0.7, label='300 ms (instant)')
ax.axvline(2000,   ls='--', color='orange', alpha=0.7, label='2 s (attention degraded)')
ax.axvline(8000,   ls='--', color='red',    alpha=0.7, label='8 s (abandonment risk)')
ax.axvline(10000,  ls=':',  color='black',  alpha=0.5, label='10 s (floor)')
ax.fill_between(t, 0, u, alpha=0.15, color='steelblue')
ax.set_xscale('log')
ax.set_xlabel('TTFT (ms, log scale)')
ax.set_ylabel('Utility $U(t)$')
ax.set_title('Latency-Utility Curve')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
ax.set_ylim(-0.05, 1.1)
plt.tight_layout()
plt.savefig('latency_utility_curve.png', dpi=150)
plt.show()